# Data Mining

## Preamble

### Modules

In [1]:
import pandas as pd

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

from utils import load_and_drop_na_data

### Loading Data

In [2]:
df = load_and_drop_na_data('../data/transformed/transformed.csv')

Raw data: 406585 rows
Removed 5223 duplicates
Clean data: 401362 rows


In [3]:
df.head(5)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [10]:
df.columns.tolist()

['InvoiceNo',
 'StockCode',
 'Description',
 'Quantity',
 'InvoiceDate',
 'UnitPrice',
 'CustomerID',
 'Country']

### Configuration

## Market Basket Analysis

### Preamble

The algorithm that will be used is the Apriori algorithm.

The following terms will be used:

- _itemset_: This represents a collection of one or more items.
- _transaction_: This represents an itemset purchased by a single user with an attached InvoiceNo.

### Objective

The aim of this analysis is to evaluate over multiple transactions the precursor itemset $X$ and successor itemset $Y$ and their correlation. That is, if an itemset $X$ is bought how does it affect the purchase of an itemset. The correlation can be:

- **Positive:** When $X$ is bought then $Y$ is likely to be bought in a certain percentage of transactions. That is, purchasing $X$ encourages customers to purchase $Y$. 
- **Negative:** When $X$ is bought then $Y$ is unlikely to be bought in a certian percentage of transactions. That is, purchasing $Y$ discourages customers from purchasing $Y$.
- **Neutral:** When $X$ is bought there is no influence on the purchase of $Y$.

This correlation is mapped out by three variables:

- Support
- Confidence
- Lift

Lift is used to handle the possibility of a spurious correlation between $X$ and $Y$.

### Key Metrics

#### Support

This is the fraction of transactions containing $X$:

$$\mathrm{supp}(X) = \frac{\lvert \{\,t \mid X \subseteq t\} \rvert}{\lvert \text{Transactions} \rvert}$$

For a rule $X \Rightarrow Y$:

$$\mathrm{supp}(X \cup Y)$$

#### Confidence

This is the probability that Y will occur given X:

$$\mathrm{conf}(X \Rightarrow Y) = P(Y \mid X) = \frac{\mathrm{supp}(X \cap Y)}{\mathrm{supp}(X)}$$

It focuses on reflecting how reliable the implication is.

#### Lift

It measures how much more often $X$ and $Y$ occur together than if independent.

This handles the scenario where both $X$ and $Y$ are popular. The support and confidence can make it seem that $X$ has a positive effect on $Y$ when in reality $Y$ is actually really popular and its sales are from its own popularity.

$$\mathrm{lift}(X \Rightarrow Y) = \frac{\mathrm{supp}(X \cap Y)}{\mathrm{supp}(X)\cdot \mathrm{supp}(Y)}
= \frac{\mathrm{conf}(X \Rightarrow Y)}{\mathrm{supp}(Y)}$$

- Lift > 1 -> Positive association
- Lift = 1 -> Independence
- Lift < 1 -> Negative association

Independence in this case means that $X$ has no impact on $Y$ and that they are both popular or unpopular on their own.

### Visual Illustrations

<img src="https://miro.medium.com/max/403/0*pyOADkeaWyrVP2ft.png" />

<img src='https://annalyzin.files.wordpress.com/2016/04/association-rule-support-table.png?w=503&h=447' />

<img src='https://miro.medium.com/max/576/1*50GI4dR58MnhwBP9dw6nFQ.png' />

<img src='https://annalyzin.files.wordpress.com/2016/03/association-rule-confidence-eqn.png?w=527&h=77' />

<img src='https://annalyzin.files.wordpress.com/2016/03/association-rule-lift-eqn.png?w=566&h=80' />

### Apriori Algorithm

In [4]:
apriori_df = df.dropna(subset=["InvoiceNo", "Description"])

apriori_df["Description"] = apriori_df["Description"].str.strip().str.lower()

transactions = apriori_df.groupby("InvoiceNo")["Description"].apply(list).tolist()

In [5]:
transactions[:5]

[['white hanging heart t-light holder',
  'white metal lantern',
  'cream cupid hearts coat hanger',
  'knitted union flag hot water bottle',
  'red woolly hottie white heart.',
  'set 7 babushka nesting boxes',
  'glass star frosted t-light holder'],
 ['hand warmer union jack', 'hand warmer red polka dot'],
 ['assorted colour bird ornament',
  "poppy's playhouse bedroom",
  "poppy's playhouse kitchen",
  'feltcraft princess charlotte doll',
  'ivory knitted mug cosy',
  'box of 6 assorted colour teaspoons',
  'box of vintage jigsaw blocks',
  'box of vintage alphabet blocks',
  'home building block word',
  'love building block word',
  'recipe box with metal heart',
  'doormat new england'],
 ['jam making set with jars',
  'red coat rack paris fashion',
  'yellow coat rack paris fashion',
  'blue coat rack paris fashion'],
 ['bath building block word']]

In [6]:
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
apriori_df_encoded = pd.DataFrame(te_ary, columns=te.columns_)

In [7]:
apriori_df_encoded.head()

,10 colour spaceboy pen,12 coloured party balloons,12 daisy pegs in wood box,12 egg house painted wood,12 hanging eggs hand painted,12 ivory rose peg place settings,12 message cards with envelopes,12 pencil small tube woodland,12 pencils small tube red retrospot,12 pencils small tube skull,...,zinc star t-light holder,zinc sweetheart soap dish,zinc sweetheart wire letter rack,zinc t-light holder star large,zinc t-light holder stars large,zinc t-light holder stars small,zinc top 2 door wooden shelf,zinc willie winkie candle stick,zinc wire kitchen organiser,zinc wire sweetheart letter tray
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [10]:
frequent_itemsets = apriori(apriori_df_encoded, min_support=0.05, use_colnames=True)

In [11]:
frequent_itemsets[:5]

,support,itemsets


In [ ]:
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)

In [ ]:
rules.head()

## Clustering

## Classification

## Time Series